# 03 — Validación de la capa Gold

Este notebook valida la calidad de las tablas `FactSales` y `DimCustomer` del Lakehouse `LH_Gold`.

**Compatibilidad con `00_orquestador`:** el orquestador utiliza `LH_Silver` como Lakehouse predeterminado. Por ello, este notebook obtiene la ruta ABFS de `LH_Gold` y lee las tablas Delta directamente, sin depender del Lakehouse predeterminado.

Las comprobaciones cubren recuentos, claves nulas, unicidad, integridad referencial y consistencia de importes. Si alguna regla falla, el notebook lanza una excepción para que la orquestación completa quede marcada como fallida.

## 1. Obtener la ruta de LH_Gold

In [ ]:
print("Iniciando validación de la capa Gold")

try:
    gold_root = notebookutils.lakehouse.get("LH_Gold")["properties"]["abfsPath"]
except Exception as e:
    raise RuntimeError(
        "No se pudo obtener la ruta ABFSS de LH_Gold. "
        "Comprueba que LH_Gold existe en el workspace y está accesible."
    ) from e

gold_tables_path = gold_root + "/Tables/dbo"
print("Ruta de las tablas Gold obtenida correctamente.")

## 2. Leer las tablas Gold

In [ ]:
fact_sales_df = (
    spark.read
    .format("delta")
    .load(f"{gold_tables_path}/FactSales")
)

dim_customer_df = (
    spark.read
    .format("delta")
    .load(f"{gold_tables_path}/DimCustomer")
)

print("Tablas Gold leídas correctamente.")

## 3. Validar recuentos

In [ ]:
total_sales = fact_sales_df.count()
total_customers = dim_customer_df.count()

print(f"Total ventas: {total_sales}")
print(f"Total clientes: {total_customers}")

## 4. Validar integridad referencial

In [ ]:
from pyspark.sql import functions as F

# Claves nulas.
null_sale_keys = fact_sales_df.filter(
    F.col("SaleId").isNull() | F.col("CustomerId").isNull()
).count()

null_customer_keys = dim_customer_df.filter(
    F.col("CustomerId").isNull()
).count()

# Unicidad de claves.
duplicate_sales = (
    fact_sales_df.groupBy("SaleId").count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_customers = (
    dim_customer_df.groupBy("CustomerId").count()
    .filter(F.col("count") > 1)
    .count()
)

# Integridad referencial: ventas cuyo CustomerId no existe en DimCustomer.
orphan_sales = (
    fact_sales_df
    .join(
        dim_customer_df.select("CustomerId"),
        on="CustomerId",
        how="left_anti"
    )
    .count()
)

# Reglas de negocio sobre cantidad y precio.
invalid_sales = fact_sales_df.filter(
    F.col("Quantity").isNull()
    | F.col("UnitPrice").isNull()
    | (F.col("Quantity") <= 0)
    | (F.col("UnitPrice") < 0)
).count()

# Consistencia del importe. Se admite una diferencia máxima de un céntimo
# para evitar falsos positivos por representación decimal/float.
amount_mismatches = fact_sales_df.filter(
    F.col("TotalAmount").isNull()
    | (
        F.abs(
            F.col("TotalAmount")
            - F.round(F.col("Quantity") * F.col("UnitPrice"), 2)
        ) > F.lit(0.01)
    )
).count()

print(f"Claves nulas en FactSales: {null_sale_keys}")
print(f"CustomerId nulos en DimCustomer: {null_customer_keys}")
print(f"SaleId duplicados: {duplicate_sales}")
print(f"CustomerId duplicados: {duplicate_customers}")
print(f"Ventas sin cliente asociado: {orphan_sales}")
print(f"Ventas con cantidad/precio inválidos: {invalid_sales}")
print(f"Importes inconsistentes: {amount_mismatches}")

## 5. Resultado de las nueve comprobaciones

Todas las reglas deben cumplirse para que el notebook finalice correctamente. Cualquier incumplimiento genera una excepción, lo que detiene la cadena de orquestación.

In [ ]:
# Resultado de las nueve comprobaciones de calidad.
checks = [
    ("FactSales no está vacía", total_sales > 0, f"Registros: {total_sales}"),
    ("DimCustomer no está vacía", total_customers > 0, f"Registros: {total_customers}"),
    ("FactSales no contiene claves nulas", null_sale_keys == 0, f"Registros afectados: {null_sale_keys}"),
    ("DimCustomer no contiene CustomerId nulos", null_customer_keys == 0, f"Registros afectados: {null_customer_keys}"),
    ("SaleId es único", duplicate_sales == 0, f"Claves duplicadas: {duplicate_sales}"),
    ("CustomerId es único en DimCustomer", duplicate_customers == 0, f"Claves duplicadas: {duplicate_customers}"),
    ("No hay ventas huérfanas", orphan_sales == 0, f"Ventas afectadas: {orphan_sales}"),
    ("Cantidad y precio cumplen las reglas", invalid_sales == 0, f"Ventas inválidas: {invalid_sales}"),
    ("TotalAmount coincide con Quantity × UnitPrice", amount_mismatches == 0, f"Importes inconsistentes: {amount_mismatches}"),
]

for nombre, correcto, detalle in checks:
    estado = "OK" if correcto else "ERROR"
    print(f"[{estado}] {nombre} — {detalle}")

fallidas = [nombre for nombre, correcto, _ in checks if not correcto]

if fallidas:
    raise RuntimeError(
        "VALIDACIÓN GOLD FALLIDA. Reglas incumplidas: "
        + "; ".join(fallidas)
    )

print("VALIDACIÓN GOLD COMPLETADA CORRECTAMENTE: 9/9 comprobaciones superadas.")